# 01 - EDA scratchpad (Phase 2 prototyping)
Prototype the Phase 2 SQL here **before** moving it into `sql/audit_queries.sql`.
Only `psycopg2` is used - no Pandas / Polars.

In [ ]:
import sys
sys.path.insert(0, "..")          # make config.py importable from notebooks/

import psycopg2
from config import DB_CONFIG, load_queries

conn = psycopg2.connect(**DB_CONFIG)
cur = conn.cursor()

def show(query, limit=20):
    """Run a query and print rows as a plain text table (no DataFrames)."""
    cur.execute(query)                       # no params -> '%' in SQL is safe
    header = [c[0] for c in cur.description]
    rows = cur.fetchmany(limit)
    widths = [max(len(str(h)), *(len(str(r[i])) for r in rows)) if rows else len(str(h))
              for i, h in enumerate(header)]
    print(" | ".join(str(h).ljust(w) for h, w in zip(header, widths)))
    print("-+-".join("-" * w for w in widths))
    for r in rows:
        print(" | ".join(str(v).ljust(w) for v, w in zip(r, widths)))

## 1. Baseline row counts

In [ ]:
queries = load_queries("audit_queries.sql")
show(queries["baseline_row_counts"])

## 2. Structure: columns per table

In [ ]:
show(queries["table_columns"])

## 3. Target baseline (`is_late_delivery`)

In [ ]:
show(queries["target_baseline"])

## 4. Quick look at an individual defect: missing delivery dates by order status

In [ ]:
show("""
SELECT order_status,
       COUNT(*) AS orders,
       COUNT(*) FILTER (WHERE order_delivered_customer_date IS NULL) AS no_delivery_date
FROM orders
GROUP BY order_status
ORDER BY orders DESC
""")

## 5. Duplicated review ids (regex / duplicate prototyping)

In [ ]:
show("""
SELECT review_id, COUNT(*) AS times_seen
FROM order_reviews
GROUP BY review_id
HAVING COUNT(*) > 1
ORDER BY times_seen DESC
LIMIT 10
""")

## 6. Prototype: descriptive statistics for one column

In [ ]:
show("""
SELECT COUNT(price) AS n, MIN(price), MAX(price), ROUND(AVG(price), 2) AS avg,
       ROUND(STDDEV(price), 2) AS stddev,
       PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY price) AS median
FROM order_items
""")

## 7. Full Phase 2 outputs (same SQL the pipeline runs)

In [ ]:
show(queries["quality_audit"], limit=80)

In [ ]:
cur.close()
conn.close()